# 20 - Isolation Forest with three seeds

How much do the Isolation Forest results depend on the random seed? Same protocol as notebook 14, run with three seeds on HDFS and BGL.

**Seeds: 42, 7, 123.** 42 is the seed the code has always used (`IForestDetector`); 7 and 123 are the two extra seeds, the same ones to use for the Random Forest and LSTM seed studies.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.models.unsupervised import IForestDetector, pick_best, validated_sweeps

DOCS, PROCESSED = ROOT / "docs", ROOT / "data" / "processed"
SEEDS = [42, 7, 123]
PERCENTILES = [90, 95, 97.5, 99, 99.5]
LABELLED = {"HDFS": ("hdfs", "order"), "BGL": ("bgl", "window")}
METRICS = ["precision", "recall", "f1", "false_alarm_rate"]

## Runs

Isolation Forest as now: 100 trees, `max_samples=1.0`, the unseen-event rule (`flag_unseen`). Fitted on the normal windows of the first 60%, the percentile chosen by F1 on the next 10% separately for each seed, tested once on the last 30%.

In [2]:
rows = []
for name, (prefix, order_col) in LABELLED.items():
    counts = pd.read_parquet(PROCESSED / f"{prefix}_counts.parquet")
    for seed in SEEDS:
        detector = IForestDetector(max_samples=1.0, flag_unseen=True, random_state=seed)
        val, test = validated_sweeps(counts, {"Isolation Forest": detector}, PERCENTILES, order_col)
        best = pick_best(val, test).iloc[0]
        rows.append({"dataset": name, "seed": seed, "chosen_percentile": best["percentile"], **{m: best[m] for m in METRICS}})

results = pd.DataFrame(rows)
results.round(4).to_csv(DOCS / "seed_results_iforest.csv", index=False)
results.style.format({"chosen_percentile": "{:g}"} | {m: "{:.4f}" for m in METRICS}).hide(axis="index")

dataset,seed,chosen_percentile,precision,recall,f1,false_alarm_rate
HDFS,42,99,0.7873,0.5111,0.6199,0.0030
HDFS,7,99,0.7923,0.5267,0.6328,0.0030
HDFS,123,99,0.7923,0.5267,0.6328,0.0030
BGL,42,99.5,0.0960,0.9908,0.1750,0.7522
BGL,7,99.5,0.0976,0.9906,0.1777,0.7381
BGL,123,99.5,0.0977,0.9904,0.1778,0.7375


## Mean and standard deviation over the seeds

Sample standard deviation (n - 1). The reported values are the seed-42 results from notebooks 14 and 16.

In [3]:
summary = results.groupby("dataset", sort=False)[METRICS].agg(["mean", "std"])
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()
reported = pd.read_csv(DOCS / "cross_system_results.csv").query("model == 'Isolation Forest' and dataset == `chosen on`")
summary["reported_f1"] = summary["dataset"].map(reported.set_index("dataset")["f1"])
summary.style.format({c: "{:.4f}" for c in summary.columns if c != "dataset"}).hide(axis="index")

dataset,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,false_alarm_rate_mean,false_alarm_rate_std,reported_f1
HDFS,0.7907,0.0029,0.5215,0.0090,0.6285,0.0075,0.0030,0.0000,0.6199
BGL,0.0971,0.0010,0.9906,0.0002,0.1768,0.0016,0.7426,0.0083,0.1750


In [4]:
pd.DataFrame({m: summary.set_index("dataset").apply(lambda r: f"{r[m + '_mean']:.3f} ± {r[m + '_std']:.3f}", axis=1)
              for m in METRICS})

,precision,recall,f1,false_alarm_rate
dataset,,,,
HDFS,0.791 ± 0.003,0.522 ± 0.009,0.628 ± 0.007,0.003 ± 0.000
BGL,0.097 ± 0.001,0.991 ± 0.000,0.177 ± 0.002,0.743 ± 0.008


## Findings

- **Seed 42 gives exactly the reported results** (HDFS F1 0.620, BGL F1 0.175), so nothing else in the setup changed.
- **The results are stable.** HDFS F1 is 0.629 ± 0.007, BGL F1 0.177 ± 0.002, and every seed chose the same percentile (99 on HDFS, 99.5 on BGL).
- On HDFS, seeds 7 and 123 give the same numbers to four decimals, slightly better than seed 42 (F1 0.633). HDFS has few distinct normal count vectors (209 in notebook 14), so different forests easily end up ranking the windows the same way.
- The reported (seed-42) values are the lowest of the three, about one standard deviation below the mean on both datasets (F1 differences of 0.009 and 0.002); the conclusions don't depend on the seed.